# 🔄 Notebook 4: Failure Handling

Handle crashes, timeouts, and retries gracefully.

## Learning Objectives

By the end of this notebook, you'll understand:
- Visibility timeout pattern
- Heartbeat mechanism
- Automatic retries
- Crash recovery

In [ ]:
import redis
import psycopg2
import json
import uuid
import time
import random
from datetime import datetime, timedelta
from typing import Optional

r = redis.Redis(host='localhost', port=6379, decode_responses=True)
r.flushall()

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="taskqueue", user="postgres", password="postgres"
)
conn.autocommit = True

cursor = conn.cursor()
cursor.execute("DELETE FROM job_logs")
cursor.execute("DELETE FROM dead_letter_queue")
cursor.execute("DELETE FROM jobs")
cursor.close()

print("✅ Connected to Redis and PostgreSQL!")

## 💥 The Crash Problem

In [ ]:
print("💥 What Happens When Workers Crash?")
print("=" * 60)
print("""
SCENARIO: Worker crashes while processing
─────────────────────────────────────────────────────────────

t=0:  Worker pops job from queue
      Queue: [job_2, job_3]  (job_1 removed!)
      DB: job_1 status = 'processing'

t=5:  Worker crashes! 💥
      Queue: [job_2, job_3]  (job_1 NOT here)
      DB: job_1 status = 'processing' (STUCK!)

PROBLEMS:
• Job is gone from queue (already popped)
• Job is stuck in 'processing' forever
• User never gets their report!

─────────────────────────────────────────────────────────────

SOLUTION: Visibility Timeout
─────────────────────────────────────────────────────────────
Instead of removing from queue, HIDE it temporarily.
If not acknowledged within timeout, job reappears.

t=0:  Worker claims job_1 (hidden for 30s)
t=5:  Worker crashes!
t=30: Timeout! job_1 reappears in queue
t=31: Another worker picks it up
""")

## ⏰ Visibility Timeout Implementation

In [ ]:
PENDING_QUEUE = "jobs:pending"
PROCESSING_SET = "jobs:processing"
VISIBILITY_TIMEOUT = 30

class ReliableQueue:
    def __init__(self, redis_client, visibility_timeout: int = 30):
        self.redis = redis_client
        self.timeout = visibility_timeout
    
    def push(self, job_id: str):
        self.redis.lpush(PENDING_QUEUE, job_id)
    
    def claim(self, timeout: int = 1) -> Optional[str]:
        result = self.redis.brpop(PENDING_QUEUE, timeout=timeout)
        if not result:
            return None
        
        _, job_id = result
        
        claim_time = time.time()
        self.redis.zadd(PROCESSING_SET, {job_id: claim_time})
        
        return job_id
    
    def acknowledge(self, job_id: str):
        self.redis.zrem(PROCESSING_SET, job_id)
    
    def requeue_stale(self) -> int:
        now = time.time()
        cutoff = now - self.timeout
        
        stale_jobs = self.redis.zrangebyscore(PROCESSING_SET, 0, cutoff)
        
        for job_id in stale_jobs:
            self.redis.zrem(PROCESSING_SET, job_id)
            self.redis.lpush(PENDING_QUEUE, job_id)
        
        return len(stale_jobs)
    
    def extend_timeout(self, job_id: str):
        self.redis.zadd(PROCESSING_SET, {job_id: time.time()})
    
    def stats(self) -> dict:
        return {
            'pending': self.redis.llen(PENDING_QUEUE),
            'processing': self.redis.zcard(PROCESSING_SET)
        }

queue = ReliableQueue(r, visibility_timeout=5)
print("✅ ReliableQueue ready!")

In [ ]:
print("⏰ Visibility Timeout Demo")
print("=" * 60)

print("\n1️⃣ Adding jobs to queue...")
for i in range(3):
    queue.push(f"job_{i}")
print(f"   {queue.stats()}")

print("\n2️⃣ Worker claims a job (but doesn't acknowledge)...")
claimed_job = queue.claim()
print(f"   Claimed: {claimed_job}")
print(f"   {queue.stats()}")

print("\n3️⃣ Simulating worker crash (waiting for timeout)...")
for i in range(6):
    time.sleep(1)
    requeued = queue.requeue_stale()
    if requeued:
        print(f"   t={i+1}s: ♻️ Requeued {requeued} stale job(s)!")
    else:
        print(f"   t={i+1}s: Waiting...")

print(f"\n📊 Final state: {queue.stats()}")
print("\n✅ Crashed job automatically returned to queue!")

## 💓 Heartbeat Pattern

In [ ]:
print("💓 Heartbeat Pattern")
print("=" * 60)
print("""
PROBLEM: Job takes 2 minutes, timeout is 30 seconds
─────────────────────────────────────────────────────────────
t=0:   Worker claims job
t=30:  Timeout! Job requeued (but still running!)
t=31:  Another worker claims same job
t=60:  TWO workers processing same job!

SOLUTION: Heartbeat - worker extends timeout periodically
─────────────────────────────────────────────────────────────
t=0:   Worker claims job
t=20:  💓 Heartbeat - extend timeout to t=50
t=40:  💓 Heartbeat - extend timeout to t=70
t=60:  💓 Heartbeat - extend timeout to t=90
t=75:  Job completes, acknowledged

RULE: Heartbeat interval < timeout / 2
      (e.g., 10s heartbeat for 30s timeout)
""")

In [ ]:
import threading

class HeartbeatWorker:
    def __init__(self, worker_id: str, queue: ReliableQueue):
        self.worker_id = worker_id
        self.queue = queue
        self.current_job = None
        self.heartbeat_thread = None
        self.should_stop = False
    
    def start_heartbeat(self, job_id: str, interval: int = 2):
        self.should_stop = False
        self.current_job = job_id
        
        def heartbeat_loop():
            while not self.should_stop and self.current_job:
                time.sleep(interval)
                if not self.should_stop and self.current_job:
                    self.queue.extend_timeout(self.current_job)
                    print(f"      💓 Heartbeat for {self.current_job}")
        
        self.heartbeat_thread = threading.Thread(target=heartbeat_loop)
        self.heartbeat_thread.start()
    
    def stop_heartbeat(self):
        self.should_stop = True
        self.current_job = None
        if self.heartbeat_thread:
            self.heartbeat_thread.join(timeout=1)
    
    def process_job(self, job_id: str, work_duration: int):
        print(f"   🔧 Processing {job_id} (will take {work_duration}s)...")
        self.start_heartbeat(job_id, interval=2)
        
        try:
            time.sleep(work_duration)
            self.queue.acknowledge(job_id)
            print(f"   ✅ Completed {job_id}")
        finally:
            self.stop_heartbeat()

print("💓 Heartbeat Demo")
print("=" * 60)

r.delete(PENDING_QUEUE)
r.delete(PROCESSING_SET)

queue = ReliableQueue(r, visibility_timeout=5)
queue.push("long_job_1")

print("\n⏱️ Processing 8-second job with 5-second timeout...")
print("   (Without heartbeat, it would be requeued at t=5!)")

worker = HeartbeatWorker("worker-1", queue)
job_id = queue.claim()
worker.process_job(job_id, work_duration=8)

print(f"\n📊 Final state: {queue.stats()}")
print("\n✅ Long job completed without being requeued!")

## 🔄 Retry Logic

In [ ]:
print("🔄 Retry Logic")
print("=" * 60)
print("""
Not all failures are permanent. Some deserve retries:

TRANSIENT FAILURES (should retry):
• Network timeout
• Database connection lost
• External API rate limited
• Out of memory (temporary)

PERMANENT FAILURES (don't retry):
• Invalid input data
• File not found
• Permission denied
• Business logic error

RETRY STRATEGY:
─────────────────────────────────────────────────────────────
Attempt 1: Immediate
Attempt 2: Wait 1 second
Attempt 3: Wait 4 seconds (exponential backoff)
Attempt 4: Wait 16 seconds
Attempt 5: Give up → Dead Letter Queue
""")

In [ ]:
class RetryableWorker:
    def __init__(self, conn, queue: ReliableQueue, max_attempts: int = 3):
        self.conn = conn
        self.queue = queue
        self.max_attempts = max_attempts
    
    def process_with_retry(self, job_id: str, handler) -> bool:
        cursor = self.conn.cursor()
        cursor.execute("SELECT attempts FROM jobs WHERE id = %s", (job_id,))
        row = cursor.fetchone()
        attempts = row[0] if row else 0
        
        cursor.execute("""
            UPDATE jobs SET attempts = attempts + 1, 
                           status = 'processing',
                           updated_at = NOW()
            WHERE id = %s
        """, (job_id,))
        cursor.close()
        
        try:
            result = handler()
            
            cursor = self.conn.cursor()
            cursor.execute("""
                UPDATE jobs SET status = 'completed', 
                               result = %s,
                               completed_at = NOW()
                WHERE id = %s
            """, (json.dumps(result), job_id))
            cursor.close()
            
            self.queue.acknowledge(job_id)
            return True
            
        except Exception as e:
            new_attempts = attempts + 1
            
            if new_attempts >= self.max_attempts:
                cursor = self.conn.cursor()
                cursor.execute("""
                    UPDATE jobs SET status = 'dead',
                                   error_message = %s
                    WHERE id = %s
                """, (str(e), job_id))
                cursor.close()
                self.queue.acknowledge(job_id)
                print(f"      ☠️ Max attempts reached, moving to DLQ")
                return False
            else:
                cursor = self.conn.cursor()
                cursor.execute("""
                    UPDATE jobs SET status = 'pending',
                                   error_message = %s
                    WHERE id = %s
                """, (str(e), job_id))
                cursor.close()
                
                backoff = 2 ** new_attempts
                print(f"      🔄 Retry {new_attempts}/{self.max_attempts} in {backoff}s")
                self.queue.acknowledge(job_id)
                time.sleep(backoff)
                self.queue.push(job_id)
                return False

print("✅ RetryableWorker ready!")

In [ ]:
print("🔄 Retry Demo")
print("=" * 60)

r.delete(PENDING_QUEUE)
r.delete(PROCESSING_SET)

job_id = str(uuid.uuid4())
cursor = conn.cursor()
cursor.execute("""
    INSERT INTO jobs (id, job_type, payload, status, attempts, max_attempts)
    VALUES (%s, 'flaky_job', '{}', 'pending', 0, 3)
""", (job_id,))
cursor.close()

queue = ReliableQueue(r, visibility_timeout=30)
queue.push(job_id)

fail_count = [0]
def flaky_handler():
    fail_count[0] += 1
    if fail_count[0] < 3:
        raise Exception(f"Simulated failure #{fail_count[0]}")
    return {"success": True}

worker = RetryableWorker(conn, queue, max_attempts=3)

print("\n📋 Processing job that fails twice then succeeds...")
for attempt in range(5):
    claimed = queue.claim(timeout=1)
    if not claimed:
        break
    print(f"\n   Attempt {attempt + 1}:")
    success = worker.process_with_retry(claimed, flaky_handler)
    if success:
        print("   ✅ Job completed!")
        break

cursor = conn.cursor()
cursor.execute("SELECT status, attempts, error_message FROM jobs WHERE id = %s", (job_id,))
status, attempts, error = cursor.fetchone()
cursor.close()

print(f"\n📊 Final state:")
print(f"   Status: {status}")
print(f"   Attempts: {attempts}")

## 🧪 Quick Quiz

1. **What's the difference between visibility timeout and heartbeat interval?**

2. **Why use exponential backoff for retries?**

3. **When should you NOT retry a failed job?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Visibility timeout vs heartbeat:")
print("   - Timeout: How long job is hidden from queue")
print("   - Heartbeat: How often worker says 'still alive'")
print("   - Heartbeat < timeout/2 (safety margin)")
print()
print("2. Why exponential backoff:")
print("   - Gives system time to recover")
print("   - Prevents thundering herd")
print("   - 1s, 2s, 4s, 8s... spreads out retries")
print()
print("3. Don't retry when:")
print("   - Invalid input (will fail forever)")
print("   - Permission denied (won't change)")
print("   - Business logic error")
print("   - Only retry transient failures!")

## 📚 Summary

### Key Takeaways

1. **Visibility timeout** - Hide job while processing
2. **Heartbeat** - Extend timeout for long jobs
3. **Exponential backoff** - Space out retries
4. **Max attempts** - Don't retry forever
5. **Transient vs permanent** - Know what to retry

### Next Up

In **Notebook 5**, we'll learn about Dead Letter Queues:
- Isolating poison messages
- Monitoring failures
- Manual intervention